In [1]:
%%bash
apt-get install -y git-lfs > /dev/null 2>&1
git lfs install

git clone --no-checkout --depth 1 https://github.com/spMohanty/PlantVillage-Dataset.git /kaggle/working/PlantVillage-Dataset
cd /kaggle/working/PlantVillage-Dataset
git sparse-checkout init --cone
git sparse-checkout set raw/color
git checkout



Git LFS initialized.
Your branch is up to date with 'origin/master'.


Cloning into '/kaggle/working/PlantVillage-Dataset'...


In [2]:
!pip install pyyaml tqdm -q


In [3]:
import os
for folder in [
    "/kaggle/working/src/common",
    "/kaggle/working/src/models/resnet50",
    "/kaggle/working/checkpoints/resnet50",
]:
    os.makedirs(folder, exist_ok=True)

# Create __init__.py files so Python treats them as packages
for folder in [
    "/kaggle/working/src",
    "/kaggle/working/src/common",
    "/kaggle/working/src/models",
    "/kaggle/working/src/models/resnet50",
]:
    open(f"{folder}/__init__.py", "w").close()


In [4]:
%%writefile /kaggle/working/src/common/dataset.py
import random
from torch.utils.data import Dataset
import torch
from PIL import Image
import torchvision.transforms as transforms
from pathlib import Path

PLANTVILLAGE_DIR = Path("/kaggle/working/PlantVillage-Dataset/raw/color")

class PlantDiseaseDataset(Dataset):
    def __init__(self, samples, label2idx, transform=None):
        self.samples = samples
        self.label2idx = label2idx
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label_name = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        label_idx = self.label2idx[label_name]
        return img, torch.tensor(label_idx, dtype=torch.long)

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


Writing /kaggle/working/src/common/dataset.py


In [5]:
%%writefile /kaggle/working/src/common/dataloader.py
import random
from pathlib import Path
from torch.utils.data import DataLoader
from src.common.dataset import (
    PlantDiseaseDataset, PLANTVILLAGE_DIR, train_transform, val_transform,
)

def list_files_by_class(root_dir, ext_patterns=("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG")):
    classes, samples = [], []
    if not root_dir.exists():
        print(f"Directory not found: {root_dir}. Treating as empty dataset.")
        return classes, samples
    for class_dir in sorted(root_dir.iterdir()):
        if not class_dir.is_dir():
            continue
        class_name = class_dir.name
        classes.append(class_name)
        file_list = []
        for ext in ext_patterns:
            file_list.extend(class_dir.rglob(ext))
        for p in sorted(file_list):
            samples.append((str(p), class_name))
    print(f"Found {len(classes)} classes and {len(samples)} samples in {root_dir}")
    return sorted(classes), samples

def split_samples(samples, train_frac=0.8, val_frac=0.1, test_frac=0.1, seed=42):
    assert abs(train_frac + val_frac + test_frac - 1.0) < 1e-6
    samples = samples[:]
    random.Random(seed).shuffle(samples)
    n = len(samples)
    n_train = int(train_frac * n)
    n_val   = int(val_frac * n)
    train = samples[:n_train]
    val   = samples[n_train:n_train + n_val]
    test  = samples[n_train + n_val:]
    print(f"Split {len(samples)} samples into {len(train)} train, {len(val)} val, {len(test)} test")
    return train, val, test

def create_dataloaders(batch_size=32, num_workers=2):
    pv_classes, pv_samples = list_files_by_class(PLANTVILLAGE_DIR)
    label2idx = {label: i for i, label in enumerate(pv_classes)}
    pv_train, pv_val, pv_test = split_samples(pv_samples)
    pv_train_ds = PlantDiseaseDataset(pv_train, label2idx, transform=train_transform)
    pv_val_ds   = PlantDiseaseDataset(pv_val,   label2idx, transform=val_transform)
    pv_test_ds  = PlantDiseaseDataset(pv_test,  label2idx, transform=val_transform)
    loaders = {
        "train": DataLoader(pv_train_ds, batch_size=batch_size, shuffle=True,  num_workers=num_workers),
        "val":   DataLoader(pv_val_ds,   batch_size=batch_size, shuffle=False, num_workers=num_workers),
        "test":  DataLoader(pv_test_ds,  batch_size=batch_size, shuffle=False, num_workers=num_workers),
    }
    return {"loaders": loaders, "label2idx": label2idx, "num_classes": len(pv_classes)}


Writing /kaggle/working/src/common/dataloader.py


In [6]:
%%writefile /kaggle/working/src/common/utils.py
import os, json
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm import tqdm

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in tqdm(loader, leave=False, desc="train"):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for imgs, labels in tqdm(loader, leave=False, desc="eval"):
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        total_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return total_loss / total, correct / total, all_preds, all_labels

def save_checkpoint(state, filepath):
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    torch.save(state, filepath)

def plot_training_curves(train_losses, val_losses, train_accs, val_accs, save_path):
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    epochs = range(1, len(train_losses) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    ax1.plot(epochs, train_losses, label='Train')
    ax1.plot(epochs, val_losses, label='Val')
    ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(True)
    ax2.plot(epochs, train_accs, label='Train')
    ax2.plot(epochs, val_accs, label='Val')
    ax2.set_title('Accuracy'); ax2.set_xlabel('Epoch'); ax2.legend(); ax2.grid(True)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"Training curves saved to {save_path}")

def save_json(data, filepath):
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    def _convert(obj):
        if isinstance(obj, (int, float, str, bool, type(None))): return obj
        if hasattr(obj, 'item'): return obj.item()
        if hasattr(obj, 'tolist'): return obj.tolist()
        return str(obj)
    with open(filepath, 'w') as f:
        json.dump(data, f, indent=2, default=_convert)

class EarlyStopping:
    def __init__(self, patience=5, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = float('inf')
    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            return False
        self.counter += 1
        return self.counter >= self.patience


Writing /kaggle/working/src/common/utils.py


In [7]:
%%writefile /kaggle/working/src/models/resnet50/model.py
import torch.nn as nn
from torchvision import models
from torchvision.models import ResNet50_Weights

def build_resnet50(num_classes, pretrained=True, dropout=0.5):
    weights = ResNet50_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.resnet50(weights=weights)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(nn.Dropout(p=dropout), nn.Linear(in_features, num_classes))
    return model

def freeze_backbone(model):
    for name, param in model.named_parameters():
        param.requires_grad = name.startswith('fc')

def unfreeze_top_layers(model):
    for name, param in model.named_parameters():
        if name.startswith('layer4') or name.startswith('fc'):
            param.requires_grad = True


Writing /kaggle/working/src/models/resnet50/model.py


In [8]:
from pathlib import Path

path = Path("/kaggle/working/PlantVillage-Dataset/raw/color")

ext = {".jpg", ".jpeg", ".png"}
total_images = 0
classes = []

for class_dir in sorted(path.iterdir()):
    if not class_dir.is_dir():
        continue
    count = sum(1 for f in class_dir.rglob("*") if f.suffix.lower() in ext)
    print(f"  {class_dir.name}: {count} images")
    classes.append(class_dir.name)
    total_images += count

print(f"\nTotal classes : {len(classes)}")
print(f"Total images  : {total_images}")

  Apple___Apple_scab: 630 images
  Apple___Black_rot: 621 images
  Apple___Cedar_apple_rust: 275 images
  Apple___healthy: 1645 images
  Blueberry___healthy: 1502 images
  Cherry_(including_sour)___Powdery_mildew: 1052 images
  Cherry_(including_sour)___healthy: 854 images
  Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot: 513 images
  Corn_(maize)___Common_rust_: 1192 images
  Corn_(maize)___Northern_Leaf_Blight: 985 images
  Corn_(maize)___healthy: 1162 images
  Grape___Black_rot: 1180 images
  Grape___Esca_(Black_Measles): 1383 images
  Grape___Leaf_blight_(Isariopsis_Leaf_Spot): 1076 images
  Grape___healthy: 423 images
  Orange___Haunglongbing_(Citrus_greening): 5507 images
  Peach___Bacterial_spot: 2297 images
  Peach___healthy: 360 images
  Pepper,_bell___Bacterial_spot: 997 images
  Pepper,_bell___healthy: 1478 images
  Potato___Early_blight: 1000 images
  Potato___Late_blight: 1000 images
  Potato___healthy: 152 images
  Raspberry___healthy: 371 images
  Soybean___healthy: 

In [9]:
import os, sys, random
os.chdir("/kaggle/working")
sys.path.insert(0, "/kaggle/working")

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

from src.common.dataloader import create_dataloaders
from src.common.utils import train_one_epoch, evaluate, EarlyStopping
from src.models.resnet50.model import build_resnet50, freeze_backbone

DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS       = 10
BATCH_SIZE   = 32
DROPOUT      = 0.5
PATIENCE     = 5
WEIGHT_DECAY = 1e-4
SEED         = 42

result      = create_dataloaders(batch_size=BATCH_SIZE, num_workers=2)
loaders     = result["loaders"]
num_classes = result["num_classes"]
criterion   = nn.CrossEntropyLoss()

# ── Grid ──────────────────────────────────────────────────────────────────────
learning_rates = [0.1, 0.01, 0.001]
optimizers     = ["SGD", "Adam", "AdamW"]

records    = []
all_curves = {}

for opt_name in optimizers:
    for lr in learning_rates:
        run_name = f"{opt_name}_lr{lr}"
        print(f"\n{'='*50}\n  Run: {run_name}\n{'='*50}")

        set_seed(SEED)
        model = build_resnet50(num_classes, pretrained=True, dropout=DROPOUT).to(DEVICE)
        freeze_backbone(model)
        trainable = [p for p in model.parameters() if p.requires_grad]

        if opt_name == "SGD":
            optimizer = optim.SGD(trainable, lr=lr, momentum=0.9, weight_decay=WEIGHT_DECAY)
        elif opt_name == "Adam":
            optimizer = optim.Adam(trainable, lr=lr, weight_decay=WEIGHT_DECAY)
        else:
            optimizer = optim.AdamW(trainable, lr=lr, weight_decay=WEIGHT_DECAY)

        scheduler    = CosineAnnealingLR(optimizer, T_max=EPOCHS)
        es           = EarlyStopping(patience=PATIENCE)
        best_val_acc = 0.0
        val_accs     = []

        for epoch in range(1, EPOCHS + 1):
            train_loss, train_acc = train_one_epoch(model, loaders["train"], criterion, optimizer, DEVICE)
            val_loss, val_acc, _, _ = evaluate(model, loaders["val"], criterion, DEVICE)
            scheduler.step()
            val_accs.append(val_acc)
            print(f"  Epoch {epoch:>2}/{EPOCHS} | Train {train_loss:.4f}/{train_acc:.4f} | Val {val_loss:.4f}/{val_acc:.4f}")
            if val_acc > best_val_acc:
                best_val_acc = val_acc
            if es(val_loss):
                print(f"  Early stopping at epoch {epoch}")
                break

        all_curves[run_name] = val_accs
        records.append({
            "optimizer":    opt_name,
            "lr":           lr,
            "best_val_acc": round(best_val_acc, 4),
            "epochs_run":   len(val_accs),
        })
        print(f"  -> Best val acc: {best_val_acc:.4f}")

# ── Results ───────────────────────────────────────────────────────────────────
print("\n" + "="*50)
print("  RESULTS SUMMARY")
print("="*50)
df = pd.DataFrame(records).sort_values("best_val_acc", ascending=False)
print(df.to_string(index=False))

plt.figure(figsize=(12, 5))
for run_name, curve in all_curves.items():
    plt.plot(range(1, len(curve) + 1), curve, marker='o', label=run_name)
plt.xlabel("Epoch"); plt.ylabel("Val Accuracy")
plt.title("Validation Accuracy: LR & Optimizer Comparison")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True); plt.tight_layout()
os.makedirs("checkpoints/resnet50", exist_ok=True)
plt.savefig("checkpoints/resnet50/lr_optimizer_comparison.png", dpi=150)
plt.show()
print("Plot saved to checkpoints/resnet50/lr_optimizer_comparison.png")


Found 38 classes and 54305 samples in /kaggle/working/PlantVillage-Dataset/raw/color
Split 54305 samples into 43444 train, 5430 val, 5431 test

  Run: SGD_lr0.1
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 193MB/s]


  Epoch  1/10 | Train 4.7512/0.7264 | Val 1.6926/0.8810


  Epoch  2/10 | Train 3.7853/0.8037 | Val 2.0383/0.8829


  Epoch  3/10 | Train 3.5205/0.8242 | Val 1.4490/0.9064


...

  Epoch  6/10 | Train 0.3191/0.8979 | Val 0.1783/0.9418


  Epoch  7/10 | Train 0.2995/0.9023 | Val 0.1536/0.9468


  Epoch  8/10 | Train 0.2809/0.9084 | Val 0.1531/0.9468


  Epoch  9/10 | Train 0.2748/0.9100 | Val 0.1473/0.9495


  Epoch 10/10 | Train 0.2665/0.9129 | Val 0.1471/0.9486
  -> Best val acc: 0.9495

  RESULTS SUMMARY
optimizer    lr  best_val_acc  epochs_run
    AdamW 0.010        0.9519          10
    AdamW 0.100        0.9519          10
     Adam 0.010        0.9517          10
      SGD 0.100        0.9506          10
     Adam 0.001        0.9499          10
      SGD 0.010        0.9495          10
    AdamW 0.001        0.9495          10
     Adam 0.100        0.9420          10
      SGD 0.001        0.9308          10
Plot saved to checkpoints/resnet50/lr_optimizer_comparison.png


In [10]:
import os, sys, random
os.chdir("/kaggle/working")
sys.path.insert(0, "/kaggle/working")

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

from src.common.dataloader import create_dataloaders
from src.common.utils import train_one_epoch, evaluate, save_checkpoint, plot_training_curves, save_json, EarlyStopping
from src.models.resnet50.model import build_resnet50, freeze_backbone, unfreeze_top_layers

# ── Config — update optimizer/LR based on grid search winner ─────────────────
CFG = {
    "seed": 42, "batch_size": 32, "num_workers": 2,
    "dropout": 0.5, "weight_decay": 1e-4, "early_stop_patience": 7,
    "pretrained": True, "save_dir": "checkpoints/resnet50",
    "optimizer": "AdamW",       # ← update with grid search winner
    "phase1": {"epochs": 10, "lr": 1e-2},   # ← update with winning LR
    "phase2": {"epochs": 20, "lr": 1e-3},   # ← 10x lower than phase1
}

set_seed(CFG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

result      = create_dataloaders(batch_size=CFG["batch_size"], num_workers=CFG["num_workers"])
loaders     = result["loaders"]
label2idx   = result["label2idx"]
num_classes = result["num_classes"]
print(f"Classes: {num_classes}")

model     = build_resnet50(num_classes, pretrained=CFG["pretrained"], dropout=CFG["dropout"]).to(device)
criterion = nn.CrossEntropyLoss()
history   = {"train_losses": [], "val_losses": [], "train_accs": [], "val_accs": []}
overall_best_state, overall_best_acc = None, 0.0

def make_optimizer(params, lr):
    if CFG["optimizer"] == "SGD":
        return optim.SGD(params, lr=lr, momentum=0.9, weight_decay=CFG["weight_decay"])
    elif CFG["optimizer"] == "Adam":
        return optim.Adam(params, lr=lr, weight_decay=CFG["weight_decay"])
    else:
        return optim.AdamW(params, lr=lr, weight_decay=CFG["weight_decay"])

def run_phase(phase_name, num_epochs, lr):
    global overall_best_state, overall_best_acc
    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = make_optimizer(trainable, lr)
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs)
    es        = EarlyStopping(patience=CFG["early_stop_patience"])
    best_val_acc = overall_best_acc

    for epoch in range(1, num_epochs + 1):
        train_loss, train_acc = train_one_epoch(model, loaders["train"], criterion, optimizer, device)
        val_loss, val_acc, _, _ = evaluate(model, loaders["val"], criterion, device)
        scheduler.step()
        history["train_losses"].append(train_loss)
        history["val_losses"].append(val_loss)
        history["train_accs"].append(train_acc)
        history["val_accs"].append(val_acc)
        print(f"[{phase_name}] {epoch:>2}/{num_epochs} | Train {train_loss:.4f}/{train_acc:.4f} | Val {val_loss:.4f}/{val_acc:.4f}")
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            overall_best_acc = val_acc
            overall_best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            print(f"  -> New best val acc: {best_val_acc:.4f}")
        if es(val_loss):
            print(f"  Early stopping at epoch {epoch}")
            break

freeze_backbone(model)
print(f"\n[Phase 1] Frozen backbone — optimizer: {CFG['optimizer']}, lr: {CFG['phase1']['lr']}")
run_phase("Phase1", CFG["phase1"]["epochs"], CFG["phase1"]["lr"])

unfreeze_top_layers(model)
print(f"\n[Phase 2] Fine-tuning layer4 + head — optimizer: {CFG['optimizer']}, lr: {CFG['phase2']['lr']}")
run_phase("Phase2", CFG["phase2"]["epochs"], CFG["phase2"]["lr"])

os.makedirs(CFG["save_dir"], exist_ok=True)
ckpt_path = os.path.join(CFG["save_dir"], "best_model.pth")
save_checkpoint({
    "model_state_dict": overall_best_state,
    "best_val_acc":     overall_best_acc,
    "label2idx":        label2idx,
    "num_classes":      num_classes,
}, ckpt_path)
plot_training_curves(
    history["train_losses"], history["val_losses"],
    history["train_accs"],   history["val_accs"],
    os.path.join(CFG["save_dir"], "training_curves.png"),
)
save_json(history, os.path.join(CFG["save_dir"], "history.json"))
print(f"\nDone. Best val acc: {overall_best_acc:.4f}")


Device: cuda
Found 38 classes and 54305 samples in /kaggle/working/PlantVillage-Dataset/raw/color
Split 54305 samples into 43444 train, 5430 val, 5431 test
Classes: 38

[Phase 1] Frozen backbone — optimizer: AdamW, lr: 0.01


[Phase1]  1/10 | Train 1.8051/0.7385 | Val 0.7333/0.8923
  -> New best val acc: 0.8923


[Phase1]  2/10 | Train 1.8865/0.8104 | Val 1.3783/0.8814


[Phase1]  3/10 | Train 1.9461/0.8364 | Val 0.6956/0.9267
  -> New best val acc: 0.9267


[Phase1]  4/10 | Train 1.9013/0.8448 | Val 0.8667/0.9201


...

[Phase2] 16/20 | Train 0.0035/0.9991 | Val 0.0222/0.9967


[Phase2] 17/20 | Train 0.0028/0.9991 | Val 0.0213/0.9971


[Phase2] 18/20 | Train 0.0032/0.9990 | Val 0.0208/0.9972


[Phase2] 19/20 | Train 0.0023/0.9992 | Val 0.0215/0.9974
  -> New best val acc: 0.9974


[Phase2] 20/20 | Train 0.0027/0.9992 | Val 0.0216/0.9976
  -> New best val acc: 0.9976
Training curves saved to checkpoints/resnet50/training_curves.png

Done. Best val acc: 0.9976


In [11]:
import os, sys
os.chdir("/kaggle/working")
sys.path.insert(0, "/kaggle/working")

import torch
import torch.nn as nn
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
)

from src.common.dataloader import create_dataloaders
from src.common.utils import evaluate, save_json
from src.models.resnet50.model import build_resnet50

def compute_metrics(y_true, y_pred):
    return {
        "accuracy":  accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall":    recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1":        f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }

def print_report(y_true, y_pred, class_names):
    present = sorted(set(y_true) | set(y_pred))
    names   = [class_names[i] for i in present]
    print(classification_report(y_true, y_pred, labels=present, target_names=names, zero_division=0))

def plot_confusion_matrix(y_true, y_pred, class_names, save_path, max_classes=20):
    cm        = confusion_matrix(y_true, y_pred)
    top_idx   = sorted(cm.sum(axis=1).argsort()[::-1][:max_classes])
    cm_sub    = cm[top_idx][:, top_idx]
    names_sub = [class_names[i] for i in top_idx]
    n         = len(names_sub)
    fig, ax   = plt.subplots(figsize=(max(8, n), max(6, n - 2)))
    im        = ax.imshow(cm_sub, interpolation="nearest", cmap="Blues")
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(names_sub, rotation=90, fontsize=7)
    ax.set_yticklabels(names_sub, fontsize=7)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"Confusion Matrix (top {n} of {len(class_names)} classes)")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150); plt.close()
    print(f"Confusion matrix saved to {save_path}")

device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CKPT_PATH = "checkpoints/resnet50/best_model.pth"
SAVE_DIR  = "checkpoints/resnet50"

raw_ckpt    = torch.load(CKPT_PATH, map_location=device)
label2idx   = raw_ckpt["label2idx"]
num_classes = raw_ckpt["num_classes"]
idx2label   = {v: k for k, v in label2idx.items()}
class_names = [idx2label[i] for i in range(num_classes)]

model = build_resnet50(num_classes=num_classes, pretrained=False, dropout=0.5).to(device)
model.load_state_dict(raw_ckpt["model_state_dict"])
model.eval()

result    = create_dataloaders(batch_size=32, num_workers=2)
loaders   = result["loaders"]
criterion = nn.CrossEntropyLoss()

test_loss, test_acc, all_preds, all_labels = evaluate(model, loaders["test"], criterion, device)

print(f"Test Loss : {test_loss:.4f}")
print(f"Test Acc  : {test_acc:.4f}")

metrics = compute_metrics(all_labels, all_preds)
print(f"Precision : {metrics['precision']:.4f}  (weighted)")
print(f"Recall    : {metrics['recall']:.4f}  (weighted)")
print(f"F1-Score  : {metrics['f1']:.4f}  (weighted)")

print("\n--- Per-Class Classification Report ---")
print_report(all_labels, all_preds, class_names)

plot_confusion_matrix(all_labels, all_preds, class_names,
                      save_path=os.path.join(SAVE_DIR, "confusion_matrix.png"))
save_json({
    "test_loss":          test_loss,
    "test_accuracy":      test_acc,
    "precision_weighted": metrics["precision"],
    "recall_weighted":    metrics["recall"],
    "f1_weighted":        metrics["f1"],
}, os.path.join(SAVE_DIR, "test_results.json"))
print(f"\nAll results saved to {SAVE_DIR}/")


Found 38 classes and 54305 samples in /kaggle/working/PlantVillage-Dataset/raw/color
Split 54305 samples into 43444 train, 5430 val, 5431 test


Test Loss : 0.0096
Test Acc  : 0.9978
Precision : 0.9978  (weighted)
Recall    : 0.9978  (weighted)
F1-Score  : 0.9978  (weighted)

--- Per-Class Classification Report ---
                                                    precision    recall  f1-score   support

                                Apple___Apple_scab       1.00      1.00      1.00        57
                                 Apple___Black_rot       1.00      1.00      1.00        45
                          Apple___Cedar_apple_rust       1.00      1.00      1.00        23
                                   Apple___healthy       1.00      1.00      1.00       154
                               Blueberry___healthy       1.00      1.00      1.00       155
          Cherry_(including_sour)___Powdery_mildew       1.00      1.00      1.00       116
                 Cherry_(including_sour)___healthy       1.00      1.00      1.00        93
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot       0.96      0.95      0.96        58